# Returns Prediction - improved pipeline (ROC-AUC)
Changes vs `goodone.ipynb`:
1. Cleans the dirty `category` column (60 spellings -> ~10 real categories).
2. Uses `store_id`, `product_id`, `customer_id`, `supplier_code` through **smoothed out-of-fold target encoding** (GroupKFold by `order_id`, never leave-one-out).
3. Date-only timestamp flag, order-level features, label-free entity statistics computed on train+test.
4. **Rolling time-based CV** (3 folds) instead of one holdout; encodings are rebuilt inside every fold.
5. LightGBM + CatBoost (native categoricals) + rank-average blend, final refit on all data.

Set `SAMPLE_FRAC < 1` for a fast smoke test.

In [1]:
import os, gc, time, warnings, difflib
import numpy as np, pandas as pd
import lightgbm as lgb
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
warnings.filterwarnings('ignore')
try:
    from catboost import CatBoostClassifier
    HAS_CB = True
except ImportError:
    HAS_CB = False

DATA_DIR      = os.environ.get('DATA_DIR', '.')      # folder with the csv files
SAMPLE_FRAC   = float(os.environ.get('SAMPLE_FRAC', 1.0))  # <1 => sample orders for a quick run
USE_CATBOOST  = HAS_CB and os.environ.get('USE_CATBOOST', '1') == '1'
SEED          = 42
BLEND_W_CB    = 0.4   # CatBoost weight in the rank blend; set from the CV printout (0 = LightGBM only)
N_JOBS        = -1
LGB_ROUNDS    = 3000
CB_ROUNDS     = 1500
CV_FOLDS = [('2024-04-01','2024-07-01'), ('2024-07-01','2024-10-01'), ('2024-10-01','2025-01-02')]  # (val_start, val_end)
print('CatBoost:', USE_CATBOOST, '| sample frac:', SAMPLE_FRAC)

CatBoost: True | sample frac: 1.0


## 1. Load + clean

In [2]:
P = lambda f: os.path.join(DATA_DIR, f)
tr = pd.read_csv(P('train.csv')); te = pd.read_csv(P('test.csv'))
prod = pd.read_csv(P('products.csv')); stores = pd.read_csv(P('stores.csv')); cust = pd.read_csv(P('customers.csv'))

if SAMPLE_FRAC < 1:
    keep = set(tr.order_id.drop_duplicates().sample(frac=SAMPLE_FRAC, random_state=SEED))
    tr = tr[tr.order_id.isin(keep)].reset_index(drop=True)

# --- products: fix dirty category text (case, spaces, underscores, typos) ---
prod['cat_n'] = prod.category.astype(str).str.lower().str.replace(r'[^a-z]', '', regex=True)
vc = prod.cat_n.value_counts()
canon = vc[vc >= 2000].index.tolist()                      # real categories are frequent
cmap = {c: (c if c in canon else (difflib.get_close_matches(c, canon, 1, 0.0) or [c])[0]) for c in vc.index}
prod['category_c'] = prod.cat_n.map(cmap)
print('categories:', prod.category.nunique(), '->', prod.category_c.nunique())
prod['vol'] = prod.length_cm * prod.width_cm * prod.height_cm
prod['density'] = prod.weight_kg / (prod.vol + 1e-5)
prod['price_to_cat_median'] = prod.list_price / prod.groupby('category_c').list_price.transform('median')

# --- stores / customers ---
fix_region = lambda s: s.astype(str).str.lower().str.strip().str.replace(r'[\s_-]+', '-', regex=True)
stores['region_s'] = fix_region(stores.region)
stores['is_physical'] = stores.channel.isin(['store', 'catalog']).astype(int)
cust['record_updated_at'] = pd.to_datetime(cust.record_updated_at, errors='coerce')
cust = cust.sort_values('record_updated_at').drop_duplicates('customer_id', keep='last')
cust['region_c'] = fix_region(cust.region)
cust['age_missing'] = cust.age_band.isnull().astype(int)

tr['is_train'] = 1; te['is_train'] = 0; te['returned'] = np.nan
df = pd.concat([tr, te], ignore_index=True); del tr, te; gc.collect()
df['ts'] = pd.to_datetime(df.order_ts)

df = df.merge(cust[['customer_id','region_c','customer_type','signup_date','age_band','age_missing','newsletter_opt_in','loyalty_tier']], on='customer_id', how='left')
df = df.merge(prod[['product_id','category_c','list_price','weight_kg','vol','density','price_to_cat_median','packaging_type','color','launch_date','review_count','supplier_code']], on='product_id', how='left')
df = df.merge(stores[['store_id','format','size_sqm','region_s','is_physical','manager_tenure_years','seller_rating','opened_date']], on='store_id', how='left')
print(df.shape)

categories: 60 -> 10
(1880000, 54)


## 2. Feature engineering

In [3]:
# --- time (timestamps contain many date-only placeholders at minute == 01) ---
df['hour'] = df.ts.dt.hour; df['dow'] = df.ts.dt.dayofweek; df['month'] = df.ts.dt.month
df['date_only'] = (df.ts.dt.minute == 1).astype(int)
df['is_weekend'] = df.dow.isin([5, 6]).astype(int)

# --- money / ratios ---
df['price_ratio'] = df.unit_price / df.list_price
df['line_val'] = df.unit_price * df.quantity
df['disc_abs'] = df.line_val * df.discount.fillna(0)
df['ov_ratio'] = df.order_value / (df.line_val + 1e-4)
df['freight_per_unit'] = df.freight_cost / df.quantity
df['freight_share'] = df.freight_cost / (df.order_value + 1e-4)
df['window_x_price'] = df.return_window_days * np.log1p(df.unit_price.clip(lower=0))
df['has_coupon'] = df.coupon_code.notnull().astype(int)
df['clicks_per_item'] = df.session_clicks / df.items_in_order
df['line_pos'] = df.line_no / df.items_in_order
df['cross_region'] = (df.region_c != df.region_s).astype(int)
df['days_since_signup'] = (df.ts - pd.to_datetime(df.signup_date)).dt.days
df['prod_age_days'] = (df.ts - pd.to_datetime(df.launch_date)).dt.days
df['store_age_days'] = (df.ts - pd.to_datetime(df.opened_date)).dt.days

# --- order-level context ---
g = df.groupby('order_id')
df['o_lines'] = g.order_line_id.transform('size')
df['o_qty'] = g.quantity.transform('sum')
df['o_nprod'] = g.product_id.transform('nunique')
df['qty_share'] = df.quantity / df.o_qty
df['price_rank'] = g.unit_price.rank(pct=True)
df['qty_rank'] = g.quantity.rank(pct=True)
df['val_share'] = df.line_val / g.line_val.transform('sum')

# --- label-free entity statistics (train + test together is legitimate: no target used) ---
for c in ['customer_id', 'product_id', 'store_id', 'supplier_code']:
    df[c + '_n'] = df.groupby(c)[c].transform('size')
df['cust_orders'] = df.groupby('customer_id').order_id.transform('nunique')
df['cust_mean_qty'] = df.groupby('customer_id').quantity.transform('mean')
df['cust_mean_items'] = df.groupby('customer_id').items_in_order.transform('mean')
df['prod_mean_qty'] = df.groupby('product_id').quantity.transform('mean')

# --- interaction keys used for target encoding ---
df['promo_x_channel'] = df.promo_channel_code.astype(str) + '_' + df.channel.astype(str)
df['promo_x_cat'] = df.promo_channel_code.astype(str) + '_' + df.category_c.astype(str)

# --- categorical -> integer codes (consistent across train+test) ---
CATS = ['channel','payment_method','ship_mode','device_type','promo_channel_code','warehouse_zone','picker_shift',
        'tz_hint','category_c','customer_type','age_band','loyalty_tier','format','packaging_type','color']
CB_CATS = ['customer_id', 'product_id', 'store_id', 'supplier_code']     # raw ids kept as strings for CatBoost
for c in CB_CATS: df[c + '_str'] = df[c].astype(str)
for c in CATS: df[c] = df[c].astype('category').cat.codes

BASE = ['unit_price','quantity','discount','freight_cost','order_value','line_no','items_in_order','gift_wrap',
        'return_window_days','banner_id','session_clicks','hour','dow','month']
EXTRA = ['date_only','is_weekend','price_ratio','line_val','disc_abs','ov_ratio','freight_per_unit','freight_share','window_x_price',
         'has_coupon','clicks_per_item','line_pos','cross_region','days_since_signup','prod_age_days','store_age_days',
         'list_price','weight_kg','vol','density','price_to_cat_median','review_count','size_sqm','is_physical',
         'manager_tenure_years','seller_rating','newsletter_opt_in','age_missing',
         'o_lines','o_qty','o_nprod','qty_share','price_rank','qty_rank','val_share',
         'customer_id_n','product_id_n','store_id_n','supplier_code_n','cust_orders','cust_mean_qty','cust_mean_items','prod_mean_qty']
FEATS = BASE + CATS + EXTRA
TE_COLS = ['store_id', 'product_id', 'customer_id', 'supplier_code', 'promo_x_channel', 'promo_x_cat']
KEEP = FEATS + TE_COLS + [c + '_str' for c in CB_CATS] + ['order_id', 'order_line_id', 'returned', 'is_train', 'ts']
df = df[list(dict.fromkeys(KEEP))]
for c in df.select_dtypes('float64').columns: df[c] = df[c].astype('float32')
T = df[df.is_train == 1].reset_index(drop=True); TEST = df[df.is_train == 0].reset_index(drop=True)
y_all = T.returned.astype(int).values
del df; gc.collect()
print('features:', len(FEATS), '| train', T.shape, '| test', TEST.shape)

features: 72 | train (1400005, 87) | test (479995, 87)


## 3. Out-of-fold target encoding (rebuilt inside every CV fold)

In [4]:
def target_encode(train_df, y_tr, apply_df, cols, k=30, n_splits=5, seed=SEED):
    """Smoothed mean encoding. Rows of train_df get OOF values (GroupKFold by order_id);
    apply_df (validation/test) is encoded using the full train_df. Returns two DataFrames."""
    prior = y_tr.mean()
    names = [c + '_te' for c in cols]
    oof = {n: np.full(len(train_df), prior, dtype='float32') for n in names}
    app = {}
    groups = train_df.order_id.values
    for a, b in GroupKFold(n_splits).split(train_df, y_tr, groups):
        for c in cols:
            s = pd.DataFrame({'k': train_df[c].values[a], 'y': y_tr[a]}).groupby('k').y.agg(['sum', 'count'])
            m = (s['sum'] + k * prior) / (s['count'] + k)
            oof[c + '_te'][b] = pd.Series(train_df[c].values[b]).map(m).fillna(prior).values.astype('float32')
    for c in cols:
        s = pd.DataFrame({'k': train_df[c].values, 'y': y_tr}).groupby('k').y.agg(['sum', 'count'])
        m = (s['sum'] + k * prior) / (s['count'] + k)
        app[c + '_te'] = pd.Series(apply_df[c].values).map(m).fillna(prior).values.astype('float32')
    return pd.DataFrame(oof), pd.DataFrame(app)

def build_matrix(part, te_part, feats):
    return pd.concat([part[feats].reset_index(drop=True).astype('float32'), te_part.reset_index(drop=True)], axis=1)

## 4. Rolling time-based cross-validation

In [5]:
LGB_PARAMS = dict(objective='binary', metric='auc', learning_rate=0.03, num_leaves=48, max_depth=8,
                  min_child_samples=300, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
                  lambda_l1=2.0, lambda_l2=15.0, max_bin=255, verbose=-1, num_threads=N_JOBS, seed=SEED)
CB_PARAMS = dict(iterations=CB_ROUNDS, learning_rate=0.08, depth=7, l2_leaf_reg=10, eval_metric='AUC',
                 random_seed=SEED, early_stopping_rounds=60, verbose=200, thread_count=N_JOBS)

cv_lgb, cv_cb, cv_blend, lgb_iters, cb_iters = [], [], [], [], []
for i, (v0, v1) in enumerate(CV_FOLDS, 1):
    t0 = time.time()
    tr_m = (T.ts < v0).values; va_m = ((T.ts >= v0) & (T.ts < v1)).values
    A, B = T[tr_m].reset_index(drop=True), T[va_m].reset_index(drop=True)
    yA, yB = y_all[tr_m], y_all[va_m]
    teA, teB = target_encode(A, yA, B, TE_COLS)
    XA, XB = build_matrix(A, teA, FEATS), build_matrix(B, teB, FEATS)

    m = lgb.train(LGB_PARAMS, lgb.Dataset(XA, yA), LGB_ROUNDS, valid_sets=[lgb.Dataset(XB, yB)],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    p_lgb = m.predict(XB, num_iteration=m.best_iteration); lgb_iters.append(m.best_iteration)
    cv_lgb.append(roc_auc_score(yB, p_lgb)); msg = f'fold {i} [{v0}..{v1}] LGB {cv_lgb[-1]:.4f}'
    p_bl = rankdata(p_lgb) / len(p_lgb)

    if USE_CATBOOST:
        cat_cols = CB_CATS
        XA_cb = pd.concat([XA, A[[c + '_str' for c in cat_cols]].reset_index(drop=True).rename(columns=lambda c: c.replace('_str', '_cat'))], axis=1)
        XB_cb = pd.concat([XB, B[[c + '_str' for c in cat_cols]].reset_index(drop=True).rename(columns=lambda c: c.replace('_str', '_cat'))], axis=1)
        cbm = CatBoostClassifier(**CB_PARAMS)
        cbm.fit(XA_cb, yA, cat_features=[c + '_cat' for c in cat_cols], eval_set=(XB_cb, yB), use_best_model=True)
        p_cb = cbm.predict_proba(XB_cb)[:, 1]; cb_iters.append(cbm.get_best_iteration())
        cv_cb.append(roc_auc_score(yB, p_cb)); msg += f' | CB {cv_cb[-1]:.4f}'
        p_bl = (1 - BLEND_W_CB) * p_bl + BLEND_W_CB * rankdata(p_cb) / len(p_cb)
        cv_blend.append(roc_auc_score(yB, p_bl)); msg += f' | blend {cv_blend[-1]:.4f}'
    print(msg, f'| {time.time()-t0:.0f}s')
    del A, B, XA, XB; gc.collect()

print('\nMean CV  LGB   :', np.mean(cv_lgb).round(4))
if USE_CATBOOST: print('Mean CV  CB    :', np.mean(cv_cb).round(4), '| blend:', np.mean(cv_blend).round(4))

0:	test: 0.6840297	best: 0.6840297 (0)	total: 1.09s	remaining: 27m 15s
200:	test: 0.7236469	best: 0.7259244 (156)	total: 2m 52s	remaining: 18m 34s
Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.7259244029
bestIteration = 156

Shrink model to first 157 iterations.
fold 1 [2024-04-01..2024-07-01] LGB 0.7315 | CB 0.7259 | blend 0.7305 | 289s
0:	test: 0.6393936	best: 0.6393936 (0)	total: 1.16s	remaining: 28m 54s
200:	test: 0.6913206	best: 0.6913206 (200)	total: 3m 20s	remaining: 21m 35s
Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.6914026835
bestIteration = 209

Shrink model to first 210 iterations.
fold 2 [2024-07-01..2024-10-01] LGB 0.6982 | CB 0.6914 | blend 0.6968 | 394s
0:	test: 0.5982086	best: 0.5982086 (0)	total: 1.14s	remaining: 28m 32s
200:	test: 0.6721984	best: 0.6721984 (200)	total: 3m 48s	remaining: 24m 33s
Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.6730614179
bestIteration = 248

Shrink model to first 249 ite

## 5. Final fit on all training data and submission

In [6]:
te_full, te_test = target_encode(T, y_all, TEST, TE_COLS)
X_full, X_test = build_matrix(T, te_full, FEATS), build_matrix(TEST, te_test, FEATS)

n_lgb = int(np.mean(lgb_iters) * 1.15)
preds = []
for s in range(3):                                   # 3 seeds
    p = dict(LGB_PARAMS, seed=SEED + s)
    mdl = lgb.train(p, lgb.Dataset(X_full, y_all), n_lgb)
    preds.append(rankdata(mdl.predict(X_test)) / len(X_test))
final_lgb = np.mean(preds, axis=0)
print('LightGBM done, rounds =', n_lgb)

if USE_CATBOOST:
    add = lambda part, X: pd.concat([X, part[[c + '_str' for c in CB_CATS]].reset_index(drop=True).rename(columns=lambda c: c.replace('_str', '_cat'))], axis=1)
    Xf_cb, Xt_cb = add(T, X_full), add(TEST, X_test)
    cbp = dict(CB_PARAMS); cbp.pop('early_stopping_rounds'); cbp['iterations'] = int(np.mean(cb_iters) * 1.15)
    cbm = CatBoostClassifier(**cbp)
    cbm.fit(Xf_cb, y_all, cat_features=[c + '_cat' for c in CB_CATS])
    final_cb = rankdata(cbm.predict_proba(Xt_cb)[:, 1]) / len(Xt_cb)
    final = (1 - BLEND_W_CB) * final_lgb + BLEND_W_CB * final_cb
else:
    final = final_lgb

sub = pd.DataFrame({'order_line_id': TEST.order_line_id.values, 'returned': final})
sub.to_csv('submission1.csv', index=False)
print(sub.shape, '| nulls:', int(sub.returned.isnull().sum()), '| unique ids:', sub.order_line_id.nunique())
sub.head()

LightGBM done, rounds = 1839
0:	total: 1.21s	remaining: 4m 42s
200:	total: 4m 29s	remaining: 44.2s
233:	total: 5m 18s	remaining: 0us
(479995, 2) | nulls: 0 | unique ids: 479995


,order_line_id,returned
0,L85d608999fa1ee,0.509414
1,L86272cb9543018,0.874595
2,L66b5669bdace5b,0.263480
3,L2c5886124612de,0.982294
4,L56fe7aa5ac5dd8,0.190520


## Next ideas if you want more
* Time-aware customer history (return rate over only that customer's *earlier* orders).
* Add a logistic-regression / small NN with ID embeddings as a diverse blend member.
* Optuna over `num_leaves`, `min_child_samples`, `feature_fraction` using the mean of the 3 CV folds (not a single holdout).
* Ablate feature groups with real Kaggle submissions to find anything that hurts the public score.